In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [3]:
data_protein = pd.read_csv('06_regulate_protein.groups.tsv', sep='\t', index_col=0)
data_protein_down = pd.read_csv('06_downregulate_protein.groups.tsv', sep='\t', index_col=0)
data_protein_up = pd.read_csv('06_upregulate_protein.groups.tsv', sep='\t', index_col=0)

In [4]:
data_protein.shape, data_protein_down.shape, data_protein_up.shape

((2919, 35), (79, 33), (281, 33))

In [5]:
column_onto = 'Gene Ontology (GO)'
data_protein = data_protein[~data_protein[column_onto].isna()]
data_protein_down = data_protein_down[~data_protein_down[column_onto].isna()]
data_protein_up = data_protein_up[~data_protein_up[column_onto].isna()]

In [6]:
data_protein.shape, data_protein_down.shape, data_protein_up.shape

((2466, 35), (68, 33), (215, 33))

In [7]:
gene_onto = data_protein[column_onto].apply(lambda x: x.split(';'))
gene_onto_aplanada = [item for sublista in gene_onto.to_list() for item in sublista]
values_count_gene_onto_todos = pd.DataFrame(gene_onto_aplanada).value_counts()

In [8]:
gene_onto_up = data_protein_up[column_onto].apply(lambda x: x.split(';'))
gene_onto_aplanada_up = [item for sublista in gene_onto_up.to_list() for item in sublista]
values_count_gene_onto_up = pd.DataFrame(gene_onto_aplanada_up).value_counts()

In [9]:
gene_onto_down = data_protein_down[column_onto].apply(lambda x: x.split(';'))
gene_onto_aplanada_down = [item for sublista in gene_onto_down.to_list() for item in sublista]
values_count_gene_onto_down = pd.DataFrame(gene_onto_aplanada_down).value_counts()

In [10]:
values_count_gene_onto_up
values_count_gene_onto_down
values_count_gene_onto_todos

0                                                             
cytosol [GO:0005829]                                              330
cytoplasm [GO:0005737]                                            303
 metal ion binding [GO:0046872]                                   233
plasma membrane [GO:0005886]                                      228
 ATP binding [GO:0005524]                                         228
                                                                 ... 
 protein localization to outer membrane [GO:0089705]                1
 protein kinase activity [GO:0004672]                               1
 bacterial-type flagellum-dependent cell motility [GO:0071973]      1
 bacteriocin transport [GO:0043213]                                 1
 nucleobase catabolic process [GO:0046113]                          1
Name: count, Length: 2145, dtype: int64

In [11]:
values_count_gene_onto_up.head()

0                                    
plasma membrane [GO:0005886]             31
 ATP binding [GO:0005524]                22
 ATP hydrolysis activity [GO:0016887]    21
 metal ion binding [GO:0046872]          19
cytoplasm [GO:0005737]                   16
Name: count, dtype: int64

In [12]:
N_UP = values_count_gene_onto_up.sum() 
N_DOWN = values_count_gene_onto_down.sum()
N_TOTAL = values_count_gene_onto_todos.sum()

In [13]:
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def calcular_enriquecimiento(counts_target, counts_background, n_target, n_total):
    resultados = []
    
    for term, a in counts_target.items(): # a = genes en mi lista con este GO
        
        total_go_bg = counts_background.get(term, a) 
        b = total_go_bg - a # b = genes en el resto del genoma con este GO
        
        c = n_target - a # c = genes en mi lista SIN este GO
        
        d = (n_total - n_target) - b # d = genes en el resto del genoma SIN este GO
        
        _, p_val = fisher_exact([[a, b], [c, d]], alternative='greater') # Prueba de Fisher (unilateral derecha para "enriquecimiento")
        
        resultados.append({'GO_Term': term, 'Hits': a, 'P_Value': p_val})
    
    df = pd.DataFrame(resultados)
    if not df.empty:
        df['FDR'] = multipletests(df['P_Value'], method='fdr_bh')[1]
    
    return df.sort_values('FDR')


res_up = calcular_enriquecimiento(values_count_gene_onto_up, values_count_gene_onto_todos, N_UP, N_TOTAL)


res_down = calcular_enriquecimiento(values_count_gene_onto_down, values_count_gene_onto_todos, N_DOWN, N_TOTAL)

In [14]:
res_up[res_up['FDR']< 0.05]

,GO_Term,Hits,P_Value,FDR
7,( branched-chain amino acid transmembrane tran...,10,1.875711e-09,6.171090e-07
12,"( L-amino acid transport [GO:0015807],)",6,5.516912e-06,9.075321e-04
2,"( ATP hydrolysis activity [GO:0016887],)",21,1.362160e-04,1.493836e-02
45,( glycine decarboxylation via glycine cleavage...,3,4.785464e-04,2.249168e-02
43,"( glycine cleavage complex [GO:0005960],)",3,4.785464e-04,2.249168e-02
32,"( microcin transport [GO:0042884],)",3,4.785464e-04,2.249168e-02
53,"( phosphopantetheine binding [GO:0031177],)",3,4.785464e-04,2.249168e-02
0,"(plasma membrane [GO:0005886],)",31,1.691902e-03,3.120666e-02
28,( L-valine transmembrane transporter activity ...,3,1.802208e-03,3.120666e-02
30,( long-chain fatty acid-CoA ligase activity [G...,3,1.802208e-03,3.120666e-02


In [15]:
res_down[res_down['FDR']< 0.05]

,GO_Term,Hits,P_Value,FDR
4,( reductive pentose-phosphate cycle [GO:001925...,8,8.788228e-13,1.362175e-10
6,"( protein maturation [GO:0051604],)",7,2.861474e-11,2.217643e-09
8,"( ferredoxin hydrogenase activity [GO:0008901],)",4,9.657693e-07,4.989808e-05
12,"( nickel cation binding [GO:0016151],)",4,4.708861e-06,1.824684e-04
1,"( zinc ion binding [GO:0008270],)",10,3.166235e-05,9.815327e-04
28,(carboxyl- or carbamoyltransferase activity [G...,2,9.900153e-04,2.557540e-02


In [16]:
res_up[res_up['FDR']< 0.05].to_csv('go_term_upregulate.csv')
res_down[res_down['FDR']< 0.05].to_csv('go_term_downregulate.csv')